<a href="https://colab.research.google.com/github/ekc2024/ScamGuard-MY/blob/main/Credit_Analyzer_v5_VendorGuard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/ekc2024/ScamGuard-MY/blob/main/credit-analyzer-v5-vendorguard/Credit_Analyzer_v5_VendorGuard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Analyzer v5 — VendorGuard Batch AP Automation

Lineage in [`ekc2024/ScamGuard-MY`](https://github.com/ekc2024/ScamGuard-MY):

| Version | Location | Scope |
|---|---|---|
| v2 | `credit-analyzer-complete/` | first working extractor |
| v3 | [`credit-analyzer-v3-production/`](https://github.com/ekc2024/ScamGuard-MY/blob/main/credit-analyzer-v3-production/credit-analyzer-complete/Credit_Analyzer_v3.ipynb) | single-document pipeline + PDPA masking |
| v4 | `Experian_PDPA_Credit_Analyzer_v4.ipynb` | Experian-focused branch |
| **v5** | `credit-analyzer-v5-vendorguard/` | **batch, cross-document, deterministic scoring** |

v3 analysed **one document at a time** and scored by counting risk keywords.
v5 ingests a **batch**, links the documents into one transaction, and scores with
a **deterministic weighted rules engine**.

| | v3 | v5 |
|---|---|---|
| Input | one file | a batch of files |
| Document roles | 4 | 6 (+ the 2 legacy types) |
| Cross-document checks | none | duplicate, bank mismatch, status conflict, missing PO, overdue |
| Score | keyword count heuristic | fixed weights, reproducible |
| Runs headless | no (`google.colab` import) | yes — CLI + GitHub Actions |

All logic lives in the `vendorguard` package in this repo. This notebook only
drives it, so the notebook and CI run the exact same code path.

---
## Step 1 — Setup

Works in Colab, local Jupyter, or VS Code. In Colab it clones the repo; locally
it just uses the working tree.

In [1]:
import importlib, importlib.util, os, subprocess, sys

# VG_REPO_URL lets CI and local tests point at a different remote.
REPO_URL = os.environ.get("VG_REPO_URL", "https://github.com/ekc2024/ScamGuard-MY.git")
BRANCH = os.environ.get("VG_REPO_BRANCH", "main")
PKG_SUBDIR = "credit-analyzer-v5-vendorguard"

# Clone whenever the package is not already beside us. True in a fresh Colab
# session; false when the notebook is opened inside the repo working tree.
if not os.path.isdir("vendorguard"):
    if not os.path.isdir("_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, "_repo"], check=True)
    os.chdir(os.path.join("_repo", PKG_SUBDIR))

# Install, retrying for PEP 668 environments (Codespaces, system Python).
# Never hard-fail here: the deps may already be present, and a pip policy error
# should not take the whole notebook down.
base = [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
if subprocess.run(base).returncode != 0:
    subprocess.run(base + ["--break-system-packages"])

sys.path.insert(0, os.getcwd())
importlib.invalidate_caches()

missing = [m for m in ("pdfplumber", "reportlab", "pandas")
           if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(f"[ERROR] Missing dependencies: {', '.join(missing)}. "
                     f"Install them with: pip install -r requirements.txt")

import vendorguard
print(f"[OK] vendorguard {vendorguard.__version__} loaded from {os.getcwd()}")

FileNotFoundError: [Errno 2] No such file or directory: '_repo/credit-analyzer-v5-vendorguard'

---
## Step 2 — Generate the six fictional sample PDFs

Supplier profile, purchase order, original invoice, reissued invoice, delivery
order, payment receipt. The two invoices use different layouts and different
wording on purpose — same invoice identity, different file hash.

In [ ]:
from scripts.make_samples import build

sample_paths = build("samples")
for p in sample_paths:
    print("[OK]", p)

---
## Step 3 — (Optional) Upload your own batch instead

Select several PDFs at once. Skip this cell to use the generated samples.
The `google.colab` import is guarded, so this cell no longer breaks the notebook
outside Colab.

In [ ]:
import glob, os, sys

try:
    from google.colab import files
    print("[UPLOAD] Select the PDFs for one batch (multi-select supported)...")
    uploaded = files.upload()
    if uploaded:
        os.makedirs("uploads", exist_ok=True)
        sample_paths = []
        for name, data in uploaded.items():
            path = os.path.join("uploads", name)
            with open(path, "wb") as fh:
                fh.write(data)
            sample_paths.append(path)
        sample_paths.sort()
        print(f"\n[OK] {len(sample_paths)} file(s) staged for analysis.")
except ImportError:
    print("[SKIP] Not running in Colab.")
    print("       Drop your PDFs into ./uploads and re-run, or keep the samples.")
    if glob.glob("uploads/*.pdf"):
        sample_paths = sorted(glob.glob("uploads/*.pdf"))
        print(f"       Found {len(sample_paths)} PDF(s) in ./uploads")

[UPLOAD] Select the PDFs for one batch (multi-select supported)...


---
## Step 4 — Run the batch

Eight steps in one call: store originals, classify, normalize, resolve supplier,
link documents, run checks, explain, route for review.

`review_date` is pinned so the overdue check is reproducible — leave it as `None`
to use today's date.

In [ ]:
from datetime import date
from vendorguard import analyze_batch

result = analyze_batch(sample_paths, review_date=date(2026, 8, 22))

a, txn = result["assessment"], result["transaction"]
print(f"Transaction : {txn['transaction_id']}  ({txn['supplier_name']} / {txn['supplier_id']})")
print(f"Documents   : {len(result['documents'])}\n")
for d in result["documents"]:
    print(f"  {d['doc_id']}  {d['doc_type']:<18} {d['file_name']}  sha={d['sha256'][:10]}...")
print(f"\nFindings    : {len(a['findings'])}")
for f in a["findings"]:
    print(f"  +{f['weight']:<3} {f['title']}")
print(f"\nRisk score  : {a['risk_score']}/100 ({a['risk_band']})")
print(f"Decision    : {a['decision']}")

---
## Step 5 — Render the review dashboard

Same renderer the CLI and GitHub Actions use, so the notebook view and the
published artifact are identical.

In [ ]:
from IPython.display import HTML, display
from vendorguard import render_html, write_html

os.makedirs("out", exist_ok=True)
write_html(result, "out/report.html")
display(HTML(render_html(result)))

---
## Step 6 — Inspect the evidence behind each point

Nothing in the score is a black box: every finding lists the document IDs and
field values that made it fire.

In [ ]:
for f in result["assessment"]["findings"]:
    print(f"\n[{f['weight']:+d}] {f['title']}  ({f['code']})")
    for e in f["evidence"]:
        print(f"      - {e}")

print("\n--- Audit trail ---")
for e in result["audit_trail"]:
    print(f"  Step {e['step']}: {e['action']} -> {e['result'][:90]}")

---
## Step 7 — Export

`result.json` is the machine-readable record; `report.html` is the human one.
Both are what the GitHub Actions run uploads as artifacts.

In [ ]:
from vendorguard import to_json

with open("out/result.json", "w", encoding="utf-8") as fh:
    fh.write(to_json(result))

print("Wrote out/result.json and out/report.html")
print("\nPDPA:", result["pdpa_notice"])

---
## Step 8 — Add a real LLM (optional, and deliberately outside the scoring path)

The score stays deterministic. A model is only useful here for writing the
reviewer-facing narrative, and it should never be allowed to move the number —
otherwise the same batch stops scoring the same twice.

```python
!pip install anthropic -q
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def narrate(result):
    a = result["assessment"]
    facts = "\n".join(
        f"- {f['title']} (+{f['weight']}): " + "; ".join(f["evidence"])
        for f in a["findings"]
    )
    msg = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        messages=[{"role": "user", "content": (
            "Write a short AP reviewer note for a Malaysian finance team. "
            "Use only the findings below. Do not invent facts and do not restate "
            f"the score.\n\nScore: {a['risk_score']}/100 ({a['risk_band']})\n{facts}"
        )}],
    )
    return "".join(b.text for b in msg.content if b.type == "text")

print(narrate(result))
```

## Running this outside the notebook

```bash
python scripts/make_samples.py samples
python -m vendorguard.cli "samples/*.pdf" --out out --review-date 2026-08-22
python -m pytest tests/ -q
```

GitHub renders this notebook but cannot execute it. The Actions workflow in
`.github/workflows/vendorguard-demo.yml` runs the same pipeline on every push
and publishes `report.html` — see the README.